# DATA PREPARATION SCRIPTS - CHIPATA CITY COUNCIL DATASETS

## A. BUDGET ALLOCATION SCRIPTS

## Setup
Installing `pymupdf` for extracting text directly from the budget PDF.

In [ ]:
!pip install pymupdf -q

## Load source documents
Uploading the Chipata City Council 2024 Output-Based Annual Budget and the
2025 Bi-annual Budget Performance report — the two source PDFs for the
budget allocations table.

In [ ]:
from google.colab import files
uploaded = files.upload()

## Confirm uploaded filenames
Checking the exact filenames Colab saved, since upload can alter naming
(e.g. appended characters) — needed to reference the files correctly in code.

In [ ]:
import os
print(os.listdir())

## Standardise filename
Renaming the budget PDF to a simpler, consistent name for use throughout
the notebook, since the uploaded filename may include extra characters.

In [ ]:
files_here = [f for f in os.listdir() if 'Chipata' in f and 'budget' in f]
if files_here:
    os.rename(files_here[0], "budget_2024.pdf")
print(os.listdir())

## Inspect document structure
Opening the budget PDF and extracting text from page 1 to confirm the file
has a native text layer (not scanned) and to understand its structure
before writing extraction logic.

In [ ]:
import pymupdf

doc = pymupdf.open("budget_2024.pdf")
print(f"Total pages: {len(doc)}")
print("--- PAGE 1 TEXT ---")
print(doc[0].get_text())

## Locate the budget tables
Scanning all pages to identify where the actual financial data sits.
Pages 1–2 contain mandate/strategy narrative; pages 3–4 cover revenue
(handled separately in the revenue_streams table); page 8 contains the
master Budget Allocation by Programme and Sub-Programme table (Table 3);
pages 11 onward break this down per individual programme (Constituency
Development, Local Governance, IDP, etc.) with Approved figures for
2022, 2023, and 2024.

In [ ]:
for i in range(len(doc)):
    text = doc[i].get_text()
    print(f"=== Page {i+1} ({len(text)} chars) ===")
    print(text[:200])
    print()

## Extract full text from key tables
Pulling the complete text of page 8 (master Budget Allocation by Programme
and Sub-Programme summary) and page 11 (first individual programme
breakdown — Constituency Development) to examine how figures, years, and
programme names are laid out before writing the parser.

In [ ]:
print("=== PAGE 8 FULL TEXT ===")
print(doc[7].get_text())  # index 7 = page 8

print("\n\n=== PAGE 11 FULL TEXT ===")
print(doc[10].get_text())  # index 10 = page 11

## Parse Table 3 — Budget Allocation by Programme and Sub-Programme (page 8)
The PDF's table structure extracts as flat text with columns out of visual order
(a known pymupdf limitation on multi-row-header tables). Each programme/sub-programme
is a fixed 7-line block: name, code, then 5 numeric values. In this document, only
one value per row is a real figure — the rest are placeholders ("(0)" or "-") since
prior-year comparative data isn't populated in this budget cycle. We extract that
single real value as the 2024 approved budget amount. Rows are flagged
`needs_review` since the 2022/2023 column mapping isn't reliably determined from
text order alone — this will need spot-checking against the source PDF.

In [ ]:
import pandas as pd

page8_text = doc[7].get_text()
lines = [l.strip() for l in page8_text.split('\n') if l.strip()]

# Data starts right after the header line "Estimate"
start_idx = lines.index('Estimate') + 1
data_lines = lines[start_idx:]

rows = []
i = 0
while i <= len(data_lines) - 7:
    name = data_lines[i]
    code = data_lines[i+1]

    # sanity check: code should be a short number (1-3 digits)
    if not code.replace('.', '').isdigit() or len(code) > 3:
        i += 1
        continue

    raw_values = data_lines[i+2:i+7]
    numeric_values = []
    for v in raw_values:
        v_clean = v.replace(',', '').replace('(0)', '0').replace('-', '0').strip()
        try:
            numeric_values.append(float(v_clean))
        except ValueError:
            numeric_values.append(0.0)

    approved_2024 = max(numeric_values) if numeric_values else 0.0

    rows.append({
        'record_id': f"BUD-2024-{code}",
        'council_name': 'Chipata City Council',
        'budget_year': 2024,
        'budget_period': 'annual',
        'vote_head_code': code,
        'programme_name': name,
        'sub_programme': '',
        'budget_category': 'other',
        'approved_budget_zmw': approved_2024,
        'source_document': 'budget_2024.pdf',
        'source_page': 8,
        'extraction_method': 'native_text',
        'data_quality_flag': 'needs_review',
        'notes': 'Column order inferred from single non-zero value per row; verify against source PDF'
    })
    i += 7

df_page8 = pd.DataFrame(rows)
print(f"Extracted {len(df_page8)} rows")
df_page8

## Fix duplicate record IDs and mark programme vs sub-programme
Vote head codes reset within each parent programme (e.g. code "001" appears
under multiple different programmes), so `record_id` built from code alone
isn't unique. Rebuilding it using a running row index instead. Also flagging
whether each row is a top-level programme (single-digit code, e.g. "1", "2")
or a sub-programme (3-digit code, e.g. "779", "044") based on code length,
since the source table nests sub-programmes under their parent programme.

In [ ]:
df_page8['record_id'] = [f"BUD-2024-{i:03d}" for i in range(1, len(df_page8) + 1)]

df_page8['programme_level'] = df_page8['vote_head_code'].apply(
    lambda c: 'programme' if len(c.strip()) <= 2 else 'sub_programme'
)

# quick sanity check
print(df_page8[['record_id', 'vote_head_code', 'programme_name', 'programme_level', 'approved_budget_zmw']].to_string())

## Check for LGEF-specific content
The assignment brief explicitly requires "LGEF utilisation" data. Searching
the full document for any mentions of LGEF (Local Government Equalisation
Fund) to confirm whether it appears as a distinct budget line or funding
source that needs separate tagging.

In [ ]:
for i in range(len(doc)):
    text = doc[i].get_text()
    if 'LGEF' in text.upper() or 'EQUALISATION' in text.upper():
        print(f"Page {i+1} mentions LGEF:")
        idx = text.upper().find('LGEF') if 'LGEF' in text.upper() else text.upper().find('EQUALISATION')
        print(text[max(0,idx-100):idx+300])
        print("---")

## LGEF finding
Page 4 shows LGEF as a revenue line item (K15.2M approved for 2024) under
"Grants" — this is the council's LGEF *income*, which belongs in the revenue
streams table, not budget allocations.
Page 8's programme-level budget table doesn't break down funding source per
programme, so a true "LGEF utilisation by programme" breakdown isn't
extractable from this document. Flagging this gap rather than fabricating
a mapping — noting it in the Data Description Paper's limitations section.

## Load and inspect bi-annual performance report
Opening the second source document to check its structure before extracting
actual expenditure figures, which will let us calculate budget variance
per programme.

In [ ]:
doc2 = pymupdf.open("2025-Bi-annual-Budget-Perfomance.pdf")
print(f"Total pages: {len(doc2)}")
for i in range(min(5, len(doc2))):
    print(f"=== Page {i+1} ===")
    print(doc2[i].get_text()[:400])
    print()

## Confirm scan status — OCR required
Native text extraction on the bi-annual performance PDF returns garbled/unreadable
characters, indicating this file is scanned (image-based), not text-native,
despite earlier assumptions. OCR is required to extract this document.

## Set up OCR pipeline for bi-annual performance report
The bi-annual budget performance PDF is scanned/image-based, so native text
extraction fails. Installing Tesseract OCR and pdf2image to convert each
page to an image and extract text via OCR instead. This will let us pull
actual expenditure figures to calculate budget variance per programme,
strengthening the "LGEF utilisation" and comprehensiveness requirements
in the assignment brief.

In [ ]:
!apt-get install -y tesseract-ocr poppler-utils -q
!pip install pytesseract pdf2image -q

## Convert scanned pages to images and run OCR
Converting each page of the bi-annual performance PDF to a high-resolution
image, then extracting text via Tesseract OCR. Starting with page 1 only
to verify OCR quality before running it across all 8 pages.

In [ ]:
from pdf2image import convert_from_path
import pytesseract

images = convert_from_path("2025-Bi-annual-Budget-Perfomance.pdf", dpi=300, first_page=1, last_page=1)
ocr_text = pytesseract.image_to_string(images[0])
print(ocr_text)

## Visually inspect the scanned page
OCR output on page 1 was mostly garbled. Displaying the raw converted image
to check for rotation, skew, or resolution issues before adjusting the OCR
pipeline.

In [ ]:
images[0]

## Rotate image and re-run OCR
Page was scanned rotated (confirmed via visual inspection), which degrades
OCR accuracy significantly even on a clear, high-quality scan. Rotating
90° counter-clockwise corrects the orientation before extracting text.

In [ ]:
# Rotate 90° counter-clockwise to correct the scan orientation
rotated = images[0].rotate(-90, expand=True)
rotated

In [ ]:
ocr_text = pytesseract.image_to_string(rotated)
print(ocr_text)

## OCR successful — check document scope
Rotation fix resolved OCR quality significantly. However, this page shows
revenue performance (budget vs. actual receipts by revenue source: Local
Taxes, Fees & Charges, Grants), not programme-level expenditure. This maps
more directly to the revenue_streams table than budget_allocations.
Checking remaining pages before deciding final placement.

In [ ]:
# Process all 8 pages now that rotation angle is confirmed
all_images = convert_from_path("2025-Bi-annual-Budget-Perfomance.pdf", dpi=300)

for i, img in enumerate(all_images):
    rotated_img = img.rotate(-90, expand=True)
    text = pytesseract.image_to_string(rotated_img)
    print(f"=== PAGE {i+1} ===")
    print(text[:400])
    print()

## Fix programme/sub-programme structure and infer budget category
Restructuring so `programme_name` always holds the parent programme (e.g.
"Constituency Development"), and `sub_programme` holds the child line item
name (e.g. "Community Projects") where applicable — matching the schema's
intended structure. Also inferring `budget_category` from programme context:
CDF-related programmes are tagged `cdf`, core administrative/support
programmes as `recurrent_expenditure`, and infrastructure/asset-heavy
programmes as `capital_expenditure` where the source text confirms this
(e.g. page 6 states Housing and Community Amenities is largely capital
assets). Rows without a clear signal are left as `other` and flagged for
manual review.

In [ ]:
# Step 1: forward-fill parent programme name into sub-programme rows
current_parent = None
fixed_programme_name = []
fixed_sub_programme = []

for _, row in df_page8.iterrows():
    if row['programme_level'] == 'programme':
        current_parent = row['programme_name']
        fixed_programme_name.append(row['programme_name'])
        fixed_sub_programme.append('')
    else:  # sub_programme
        fixed_programme_name.append(current_parent)
        fixed_sub_programme.append(row['programme_name'])

df_page8['programme_name'] = fixed_programme_name
df_page8['sub_programme'] = fixed_sub_programme

# Step 2: infer budget_category based on parent programme context
category_map = {
    'Constituency Development': 'cdf',
    'Management and support  Services': 'recurrent_expenditure',
    'Resource Mobilisation and Management': 'recurrent_expenditure',
    'Housing and Community Amenities': 'capital_expenditure',
    'Local Governance': 'recurrent_expenditure',
    'District Health servcies': 'recurrent_expenditure',
}

df_page8['budget_category'] = df_page8['programme_name'].map(category_map).fillna('other')

# quick check
print(df_page8[['programme_name', 'sub_programme', 'vote_head_code', 'budget_category', 'approved_budget_zmw']].to_string())

## Validate total against stated budget figure
Cross-checking the sum of top-level programme approved amounts against the
K150.2 million total stated in the narrative (page 5), to confirm whether
page 8's table was fully captured or continues beyond what was extracted.

In [ ]:
programme_total = df_page8[df_page8['sub_programme'] == '']['approved_budget_zmw'].sum()
print(f"Sum of top-level programmes: {programme_total:,.2f} ZMW")
print(f"Stated total in document: ~150,200,000 ZMW")
print(f"Difference: {150_200_000 - programme_total:,.2f} ZMW")

## Check for table continuation on page 9
Page 8's extracted text ends abruptly after "Primary Health Services,"
suggesting the table may continue onto page 9 with additional programmes
or sub-programmes not yet captured. Checking page 9's full text to confirm.

In [ ]:
print(doc[8].get_text())  # index 8 = page 9

## Extract remaining rows from page 9 and merge
Page 8's table continues onto page 9 after the narrative paragraph, with
3 additional programmes/sub-programmes (District Health Coordination,
Veterinary services, Transport Services) not yet captured. The page also
confirms a Head Total of 150,164,034 ZMW, matching the narrative figure
and validating completeness once these rows are added. Manually adding
these since the table fragment is small and mixed with narrative text
(same parsing logic as page 8 would misfire on this fragment).

In [ ]:
extra_rows = [
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '002', 'programme_name': 'District Health servcies',
     'sub_programme': 'District Health Coordination', 'budget_category': 'recurrent_expenditure',
     'approved_budget_zmw': 3968017.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '14', 'programme_name': 'Veterinary services',
     'sub_programme': '', 'budget_category': 'other',
     'approved_budget_zmw': 189173.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '001', 'programme_name': 'Veterinary services',
     'sub_programme': 'Animal Health Extension Services', 'budget_category': 'other',
     'approved_budget_zmw': 189173.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '15', 'programme_name': 'Transport Services',
     'sub_programme': '', 'budget_category': 'other',
     'approved_budget_zmw': 3708668.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '001', 'programme_name': 'Transport Services',
     'sub_programme': 'Road Transport', 'budget_category': 'other',
     'approved_budget_zmw': 3708668.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
]

import pandas as pd
df_page8 = pd.concat([df_page8, pd.DataFrame(extra_rows)], ignore_index=True)

# rebuild record_id sequentially for all rows
df_page8['record_id'] = [f"BUD-2024-{i:03d}" for i in range(1, len(df_page8) + 1)]

# re-validate total
top_level_total = df_page8[df_page8['sub_programme'] == '']['approved_budget_zmw'].sum()
print(f"New top-level total: {top_level_total:,.2f} ZMW")
print(f"Head Total stated in document: 150,164,034.00 ZMW")
print(f"Difference: {150_164_034 - top_level_total:,.2f} ZMW")
print(f"\nTotal rows now: {len(df_page8)}")

## Improve OCR quality on the Payments page (page 2)
Re-converting page 2 at higher resolution (400 DPI) with grayscale and
contrast enhancement to improve OCR accuracy on the scanned Payments table,
since the initial pass at standard resolution produced too many blanks
and garbled values to use reliably.

In [ ]:
from PIL import ImageEnhance

# Re-convert just page 2 at higher DPI
page2_images = convert_from_path("2025-Bi-annual-Budget-Perfomance.pdf", dpi=400, first_page=2, last_page=2)
page2_rotated = page2_images[0].rotate(-90, expand=True)

# Preprocess: grayscale + boost contrast
page2_gray = page2_rotated.convert('L')
enhancer = ImageEnhance.Contrast(page2_gray)
page2_enhanced = enhancer.enhance(2.0)

page2_enhanced

## Extract text via OCR with table-optimized settings
Running Tesseract with `--psm 6` (assumes a uniform block of text, better
suited to tabular layouts than the default page-segmentation mode).

In [ ]:
ocr_text_page2 = pytesseract.image_to_string(page2_enhanced, config='--psm 6')
print(ocr_text_page2)

## Crop table region and retry OCR
Full-page OCR remains unreliable on individual payment line items due to
scan quality and dense table formatting. Cropping to isolate just the
table body (excluding headers/footers) and retrying at higher resolution
with column-oriented PSM mode, to see if this improves accuracy before
deciding whether line-item detail is extractable at all.

In [ ]:
# View full image with dimensions to identify crop coordinates
width, height = page2_enhanced.size
print(f"Image size: {width} x {height}")
page2_enhanced

## Crop to isolate table region
Cropping the image to the estimated table boundaries (excluding header
and footer/signature areas) to reduce noise and improve OCR accuracy.

In [ ]:
width, height = page2_enhanced.size

# Adjust these fractions based on where the table actually sits
top_frac = 0.20      # <- replace with your estimate
bottom_frac = 0.75   # <- replace with your estimate

top = int(height * top_frac)
bottom = int(height * bottom_frac)

cropped = page2_enhanced.crop((0, top, width, bottom))
cropped

## Transcribe Payments table from page 2 (visual verification)
OCR on this scan remained unreliable even after rotation, contrast
enhancement, and cropping attempts. The table image is clearly legible on
visual inspection, so values are transcribed directly rather than forcing
unreliable automated extraction. This is documented as `extraction_method
= manual` with `data_quality_flag = verified`, since figures were read
directly from the source rather than inferred or OCR'd. This table uses
economic classification (Personal emoluments, Goods & services, etc.)
rather than programme classification, covering the 2025 bi-annual period
(01-Jan-2025 to 30-Jun-2025), budgeted against a full-year 2025 approved
figure.

In [ ]:
payments_rows = [
    {'sub_programme': 'Personal emoluments', 'budget_category': 'recurrent_expenditure',
     'approved_budget_zmw': 42622728.00, 'actual_expenditure_zmw': 15581875.77, 'variance_zmw': 27040852.23, 'variance_percent': 37.0},
    {'sub_programme': 'Use of goods and services', 'budget_category': 'recurrent_expenditure',
     'approved_budget_zmw': 38064123.00, 'actual_expenditure_zmw': 16889204.68, 'variance_zmw': 21174918.32, 'variance_percent': 44.0},
    {'sub_programme': 'Financial charges', 'budget_category': 'debt_service',
     'approved_budget_zmw': 0.0, 'actual_expenditure_zmw': 0.0, 'variance_zmw': 0.0, 'variance_percent': None},
    {'sub_programme': 'Social benefits', 'budget_category': 'other',
     'approved_budget_zmw': 48334175.00, 'actual_expenditure_zmw': 27800800.00, 'variance_zmw': 20533375.00, 'variance_percent': 58.0},
    {'sub_programme': 'Non-financial assets', 'budget_category': 'capital_expenditure',
     'approved_budget_zmw': 52852445.00, 'actual_expenditure_zmw': 32289463.15, 'variance_zmw': 20562981.85, 'variance_percent': 61.0},
    {'sub_programme': 'Financial assets', 'budget_category': 'other',
     'approved_budget_zmw': 7473872.00, 'actual_expenditure_zmw': 326600.00, 'variance_zmw': 7147272.00, 'variance_percent': 4.0},
    {'sub_programme': 'Loan repayments', 'budget_category': 'debt_service',
     'approved_budget_zmw': 0.0, 'actual_expenditure_zmw': 0.0, 'variance_zmw': 0.0, 'variance_percent': None},
    {'sub_programme': 'Other repayments', 'budget_category': 'other',
     'approved_budget_zmw': 3478550.00, 'actual_expenditure_zmw': 3135845.14, 'variance_zmw': 342704.86, 'variance_percent': 90.0},
]

for row in payments_rows:
    row.update({
        'council_name': 'Chipata City Council',
        'budget_year': 2025,
        'budget_period': 'bi_annual',
        'vote_head_code': '',
        'programme_name': 'Total Council (Economic Classification)',
        'source_document': '2025-Bi-annual-Budget-Perfomance.pdf',
        'source_page': 2,
        'extraction_method': 'manual',
        'data_quality_flag': 'verified',
        'notes': 'Economic classification breakdown, transcribed visually from scanned Payments table (OCR unreliable)'
    })

df_payments = pd.DataFrame(payments_rows)
df_page8 = pd.concat([df_page8, df_payments], ignore_index=True)
df_page8['record_id'] = [f"BUD-{i:03d}" for i in range(1, len(df_page8) + 1)]

# verify total payments matches source
total_check = df_payments['approved_budget_zmw'].sum()
print(f"Sum of transcribed payment categories: {total_check:,.2f}")
print(f"Total payments stated in document: 192,825,893.00")
print(f"Match: {abs(total_check - 192825893.00) < 1}")
print(f"\nTotal rows in budget_allocations now: {len(df_page8)}")

## Validate required columns before export
Checking that every row has a value in all schema-required columns
(record_id, council_name, budget_year, budget_period, programme_name,
budget_category, source_document, extraction_method, extracted_at) before
final export, to catch any gaps while they're still easy to fix.

In [ ]:
required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

print("Checking required columns for missing/blank values...\n")

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s):")
        print(missing[['record_id', 'programme_name', 'sub_programme']].to_string())
        print()
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete.' if all_clear else '⚠️ Some required columns need fixing before export.'}")

## Add extraction timestamp and re-validate
Populating `extracted_at` (the last required column) before final export,
then re-running the completeness check to confirm all required fields
are filled across all 60 rows.

In [ ]:
from datetime import datetime

df_page8['extracted_at'] = datetime.now().isoformat()

# re-run the check now that extracted_at exists
required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s)")
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete. Ready to export.' if all_clear else '⚠️ Fix issues before exporting.'}")

## Export final budget_allocations table
Saving the complete, validated table (60 rows, covering approved 2024
programme budgets and 2025 bi-annual actual payments by economic
classification) as the required pipe-separated CSV, following the naming
convention specified in the assignment brief.

In [ ]:
# ensure column order matches schema, keeping data_quality_flag as a documented extra
schema_columns = ['record_id', 'council_name', 'budget_year', 'budget_period', 'vote_head_code',
                   'programme_name', 'sub_programme', 'sector_cluster', 'budget_category', 'funding_source',
                   'approved_budget_zmw', 'revised_budget_zmw', 'actual_expenditure_zmw', 'variance_zmw',
                   'variance_percent', 'source_document', 'source_page', 'extraction_method',
                   'extracted_at', 'notes']

for col in schema_columns:
    if col not in df_page8.columns:
        df_page8[col] = ''

df_final = df_page8[schema_columns + ['data_quality_flag']]

df_final.to_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|", index=False, encoding="utf-8")
print(f"Exported {len(df_final)} rows.")
print(os.listdir())

## Verify export integrity
Reading the exported CSV back in to confirm the pipe separator and encoding
work correctly and no data was lost or corrupted during export.

In [ ]:
df_check = pd.read_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|")
print(f"Rows read back: {len(df_check)}")
print(f"Columns: {list(df_check.columns)}")
df_check.head()

In [ ]:
from google.colab import files
files.download("db-unza26-csc4792-chipata_budget_allocations.csv")

## Web scraping — Chipata City Council website
Visiting the council's official website to fulfil the assignment's explicit
requirement to extract data "from the respective council websites." Checking
for budget/financial documents that may be available online, in addition to
the PDFs already sourced locally.

In [ ]:
!pip install requests beautifulsoup4 -q

import requests
from bs4 import BeautifulSoup

url = "https://www.chipatacouncil.gov.zm"
response = requests.get(url, timeout=15)
print(f"Status code: {response.status_code}")

soup = BeautifulSoup(response.content, 'html.parser')
print(f"Page title: {soup.title.string if soup.title else 'No title found'}")

# List all links on the homepage
links = soup.find_all('a', href=True)
print(f"\nTotal links found: {len(links)}")
for link in links[:30]:
    print(link['href'], "|", link.get_text(strip=True)[:60])

## Web scraping — Chipata City Council website
Visiting the council's official website to fulfil the assignment's explicit
requirement to extract data "from the respective council websites." The
site returned an SSL certificate verification error (common with
government sites using outdated/misconfigured certificates) — verification
is disabled for this read-only request since no sensitive data is being
transmitted.

In [ ]:
import requests
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

url = "https://www.chipatacouncil.gov.zm"
response = requests.get(url, timeout=15, verify=False)
print(f"Status code: {response.status_code}")

soup = BeautifulSoup(response.content, 'html.parser')
print(f"Page title: {soup.title.string if soup.title else 'No title found'}")

links = soup.find_all('a', href=True)
print(f"\nTotal links found: {len(links)}")
for link in links[:30]:
    print(link['href'], "|", link.get_text(strip=True)[:60])

## Explore Finance Department and Publications pages
Two links from the homepage appear directly relevant to budget data: the
Finance Department page and the Publications page (often where councils
post budget/financial PDF documents). Visiting both to check for content
or downloadable files relevant to budget_allocations.

In [ ]:
finance_url = "https://www.chipatacouncil.gov.zm/?page_id=5692"
pubs_url = "https://www.chipatacouncil.gov.zm/?page_id=541"

for label, page_url in [("FINANCE DEPARTMENT", finance_url), ("PUBLICATIONS", pubs_url)]:
    print(f"=== {label} ({page_url}) ===")
    resp = requests.get(page_url, timeout=15, verify=False)
    page_soup = BeautifulSoup(resp.content, 'html.parser')

    # print main visible text (rough cut, first 1000 chars)
    text = page_soup.get_text(separator=' ', strip=True)
    print(text[:1000])

    # look specifically for PDF links
    pdf_links = [a['href'] for a in page_soup.find_all('a', href=True) if '.pdf' in a['href'].lower()]
    print(f"\nPDF links found: {len(pdf_links)}")
    for pdf in pdf_links:
        print(" -", pdf)
    print("\n")

## Confirm live source and check for newer documents
Cross-referencing the council's Publications page against the PDFs already
processed confirms they are the same files hosted live on the official
website (Chipata-City-Council-budget-2024.pdf, 2025-Bi-annual-Budget-Perfomance.pdf),
satisfying the requirement to extract data from the council's website
directly. The scrape also surfaced a document not previously on hand —
budget-approved-2026.pdf — downloaded here to check whether it extends
budget coverage into a third year.

In [ ]:
new_budget_url = "https://www.chipatacouncil.gov.zm/wp-content/uploads/2026/05/budget-approved-2026.pdf"

resp = requests.get(new_budget_url, timeout=20, verify=False)
with open("budget_2026.pdf", "wb") as f:
    f.write(resp.content)

print(f"Downloaded: {len(resp.content)} bytes")

doc2026 = pymupdf.open("budget_2026.pdf")
print(f"Total pages: {len(doc2026)}")
print("--- PAGE 1 TEXT ---")
print(doc2026[0].get_text())

In [ ]:
try:
    print(f"df_page8 exists with {len(df_page8)} rows")
except NameError:
    print("df_page8 does NOT exist — needs rebuilding")

try:
    print(f"doc2026 exists with {len(doc2026)} pages")
except NameError:
    print("doc2026 does NOT exist — needs rebuilding")

In [ ]:
import pymupdf

new_budget_url = "https://www.chipatacouncil.gov.zm/wp-content/uploads/2026/05/budget-approved-2026.pdf"

resp = requests.get(new_budget_url, timeout=20, verify=False)
with open("budget_2026.pdf", "wb") as f:
    f.write(resp.content)

print(f"Downloaded: {len(resp.content)} bytes")

doc2026 = pymupdf.open("budget_2026.pdf")
print(f"Total pages: {len(doc2026)}")
print("--- PAGE 1 TEXT ---")
print(doc2026[0].get_text())

## Check scan status of 2026 budget
Page 1 returned no extractable text, and the file size (38MB for 57 pages)
is unusually large for a text-native PDF — both signs point to this being
a scanned document. Checking a few more pages to confirm before deciding
whether to pursue OCR/manual transcription or treat this as a
lower-priority addition given the size.

In [ ]:
for i in range(min(5, len(doc2026))):
    text = doc2026[i].get_text()
    print(f"=== Page {i+1} ({len(text)} chars) ===")
    print(text[:200] if text else "(no extractable text)")
    print()

## Quick check — locate summary table in 2026 budget
Before committing to full OCR across 57 pages, checking whether the 2026
budget follows the same structure as the 2024 version (programme summary
table typically appearing around page 8-9). Converting just a few candidate
pages to images and running OCR to confirm before deciding on full
extraction scope.

In [ ]:
from pdf2image import convert_from_path
import pytesseract

# Check pages 7-10 (covers the range where the table sat in the 2024 doc)
candidate_pages = convert_from_path("budget_2026.pdf", dpi=300, first_page=7, last_page=10)

for i, img in enumerate(candidate_pages):
    page_num = i + 7
    # try both orientations since we don't know if this scan is rotated
    text_normal = pytesseract.image_to_string(img)
    text_rotated = pytesseract.image_to_string(img.rotate(-90, expand=True))

    print(f"=== PAGE {page_num} (normal) ===")
    print(text_normal[:300])
    print(f"\n=== PAGE {page_num} (rotated -90) ===")
    print(text_rotated[:300])
    print("\n" + "="*50 + "\n")

## OCR the full 2026 programme budget table
Pages 8-10 confirm the same programme/sub-programme table structure as the
2024 document, with the 2024 approved figures matching exactly — confirming
this document extends the same table with 2025 and 2026 columns. Running
OCR across the full table range (pages 8-11) to capture all three years'
figures, extending the budget_allocations table with newer approved and
estimated budget data.

In [ ]:
table_pages = convert_from_path("budget_2026.pdf", dpi=400, first_page=8, last_page=12)

full_table_text = []
for i, img in enumerate(table_pages):
    text = pytesseract.image_to_string(img, config='--psm 6')
    full_table_text.append(text)
    print(f"=== PAGE {i+8} ===")
    print(text)
    print("\n")

## Extract 2025 and 2026 programme-level budgets
Table:2 on page 8 of the 2026 document is a clean summary table showing
approved budgets for 2024, 2025, and 2026 at the programme level. The 2024
column closely matches the figures already validated from the original
2024 document (minor variance likely reflects a later restatement in this
newer publication). The detailed sub-programme breakdown on pages 9-10 has
significant OCR degradation and is not extracted at this level of detail —
only the programme-level 2025 and 2026 figures are added, transcribed and
corrected from OCR against the visible table structure.

In [ ]:
# Programme-level data for 2025 and 2026, transcribed from Table:2 (page 8)
# Format: (vote_head_code, programme_name, budget_2025, budget_2026)
programme_2025_2026 = [
    ('1', 'Constituency Development', 72116301, 80065100),
    ('2', 'Local Governance', 3536881, 3773018),
    ('3', 'Integrated Development Planning', 6187954, 7173821),
    ('4', 'Economic and business development', 2110622, 2609043),
    ('5', 'Public health and Environmental protection', 4546772, 7847518),
    ('6', 'Housing and Community Amenities', 15687300, 17904197),
    ('7', 'Recreation Culture and Religion', 267068, 267069),
    ('8', 'Education and skills development', 130251, 130925),
    ('10', 'Public order and safety', 7182633, 6620906),
    ('11', 'Management and support  Services', 31284866, 31702297),
    ('12', 'Resource Mobilisation and Management', 4806840, 5535656),
    ('13', 'District Health servcies', 7790285, 8052415),
    ('15', 'Transport Services', 3200587, 6351350),
    ('16', 'Agricultural Services', 516367, 508368),
    ('17', 'Fisheries and Livestock', 486160, 503176),
    ('18', 'Social Protection and Community Development', 32975006, 13582709),
]

category_map_extended = {
    'Constituency Development': 'cdf',
    'Management and support  Services': 'recurrent_expenditure',
    'Resource Mobilisation and Management': 'recurrent_expenditure',
    'Housing and Community Amenities': 'capital_expenditure',
    'Local Governance': 'recurrent_expenditure',
    'District Health servcies': 'recurrent_expenditure',
}

new_rows = []
for code, name, amt_2025, amt_2026 in programme_2025_2026:
    for year, amt in [(2025, amt_2025), (2026, amt_2026)]:
        new_rows.append({
            'council_name': 'Chipata City Council',
            'budget_year': year,
            'budget_period': 'annual',
            'vote_head_code': code,
            'programme_name': name,
            'sub_programme': '',
            'budget_category': category_map_extended.get(name, 'other'),
            'approved_budget_zmw': amt,
            'source_document': 'budget_2026.pdf',
            'source_page': 8,
            'extraction_method': 'manual',
            'data_quality_flag': 'needs_review',
            'notes': 'Programme-level only; sub-programme detail not extracted due to OCR degradation on pages 9-10'
        })

df_new_years = pd.DataFrame(new_rows)
df_page8 = pd.concat([df_page8, df_new_years], ignore_index=True)
df_page8['record_id'] = [f"BUD-{i:03d}" for i in range(1, len(df_page8) + 1)]

print(f"Added {len(df_new_years)} new rows (2025 + 2026 programme budgets)")
print(f"Total rows now: {len(df_page8)}")

# validate against stated totals for each year
total_2025 = df_page8[(df_page8['budget_year']==2025) & (df_page8['sub_programme']=='')]['approved_budget_zmw'].sum()
total_2026 = df_page8[(df_page8['budget_year']==2026) & (df_page8['sub_programme']=='')]['approved_budget_zmw'].sum()
print(f"\n2025 total: {total_2025:,.2f} (document shows ~192,825,892 - note likely OCR misread first digit)")
print(f"2026 total: {total_2026:,.2f} (document Head Total: 192,627,565)")

## Cross-document validation
The 2025 programme-level total extracted from the 2026 budget document
(192,825,893.00) exactly matches the "Total payments" figure independently
transcribed from the 2025 bi-annual performance report — strong
cross-document validation. The 2026 total (192,627,568.00) is within 3 ZMW
of the document's own stated Head Total (192,627,565), consistent with
rounding across 16 programmes.

In [ ]:
from datetime import datetime
df_page8['extracted_at'] = datetime.now().isoformat()

required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s)")
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete. Ready to export.' if all_clear else '⚠️ Fix issues before exporting.'}")
print(f"Total rows: {len(df_page8)}")

## Export final budget_allocations table
Saving the complete, validated table (92 rows, covering approved programme
budgets for 2024, 2025, and 2026, plus actual 2025 payments by economic
classification) as the required pipe-separated CSV, following the naming
convention specified in the assignment brief.

In [ ]:
schema_columns = ['record_id', 'council_name', 'budget_year', 'budget_period', 'vote_head_code',
                   'programme_name', 'sub_programme', 'sector_cluster', 'budget_category', 'funding_source',
                   'approved_budget_zmw', 'revised_budget_zmw', 'actual_expenditure_zmw', 'variance_zmw',
                   'variance_percent', 'source_document', 'source_page', 'extraction_method',
                   'extracted_at', 'notes']

for col in schema_columns:
    if col not in df_page8.columns:
        df_page8[col] = ''

df_final = df_page8[schema_columns + ['data_quality_flag']]

df_final.to_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|", index=False, encoding="utf-8")
print(f"Exported {len(df_final)} rows.")
print(os.listdir())

In [ ]:
df_check = pd.read_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|")
print(f"Rows read back: {len(df_check)}")
print(f"Columns: {list(df_check.columns)}")
df_check.head()

In [ ]:
from google.colab import files
files.download("db-unza26-csc4792-chipata_budget_allocations.csv")

## Remove data_quality_flag column
Dropping the data_quality_flag column to match the official schema exactly,
consistent with the other tables in the dataset. Quality-related context
for flagged rows remains preserved in the notes column where applicable.

In [ ]:
# preserve any quality flag info into notes before dropping, so nothing is lost
df_final['notes'] = df_final.apply(
    lambda row: (row['notes'] + ' | ' if row['notes'] else '') + f"[{row['data_quality_flag']}]"
    if row['data_quality_flag'] not in ('', 'verified') else row['notes'],
    axis=1
)

df_final_clean = df_final.drop(columns=['data_quality_flag'])

df_final_clean.to_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|", index=False, encoding="utf-8")
print(f"Exported {len(df_final_clean)} rows with {len(df_final_clean.columns)} columns")
print(list(df_final_clean.columns))

In [ ]:
try:
    print(f"df_page8 rows: {len(df_page8)}")
except NameError:
    print("df_page8 does NOT exist — session needs rebuilding")

try:
    print(f"doc2026 pages: {len(doc2026)}")
except NameError:
    print("doc2026 does NOT exist")

import os
print("\nFiles:", os.listdir())

In [ ]:
from datetime import datetime
df_page8['extracted_at'] = datetime.now().isoformat()

required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s)")
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete. Ready to export.' if all_clear else '⚠️ Fix issues before exporting.'}")
print(f"Total rows: {len(df_page8)}")

In [ ]:
df_check = pd.read_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|")
print(f"Final check — Rows: {len(df_check)}, Columns: {len(df_check.columns)}")

from google.colab import files
files.download("db-unza26-csc4792-chipata_budget_allocations.csv")

## B. REVENUE STREAMS EXTRACTION SCRIPTS

# Chipata City Council Revenue Streams Extraction

**Output table:** `db-unza26-csc4792-chipata_revenue_streams.csv`  
**Source folder:** `Budgets & Financial Statement`  
**Council:** Chipata City Council

This notebook extracts locally generated revenue and related income streams from Chipata City Council financial statements and budget performance reports, then exports a submission-ready pipe-separated CSV.

### Source documents used
| File | Year(s) | Content |
|------|---------|--------|
| `FINANCIAL-STATEMENT.pdf` | 2023 | Detailed revenue notes + budget vs actual variances |
| `FINANCIAL-STATEMENTS-FOR-THE-YEAR-ENDED-31TH.pdf` | 2022 | Detailed revenue notes |
| `1.-Chipata-City-Council-Bi-Annual-2025-Draft.pdf` | 2025 (Jan–Jun) | Semi-annual cash receipt summary |
| `SEMI-ANNUAL-BUDGET-EXECUTION-REPORT-FOR-2024.pdf` | 2024 | **OCR** — budget vs collected by revenue code |

### OCR prerequisites (Windows)
```powershell
winget install UB-Mannheim.TesseractOCR
winget install oschwartz10612.Poppler
```


## 1. Setup

In [ ]:
from __future__ import annotations

import re
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd
import pymupdf
import yaml

PROJECT_ROOT = Path("..").resolve()
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.revenue_pdf_parser import parse_financial_statement_year
from src.revenue_ocr_parser import parse_budget_execution_rows
RAW_TEXT_DIR = PROJECT_ROOT / "data" / "raw_text"
RAW_TEXT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_ROOT = (
    PROJECT_ROOT
    / "csc_4792_dataset_docs-20260911T121615Z-1-001"
    / "csc_4792_dataset_docs"
    / "Budgets & Financial Statement"
)
SCHEMAS_PATH = PROJECT_ROOT / "schemas" / "csv_schemas.yaml"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TABLE_NAME = "db-unza26-csc4792-chipata_revenue_streams"
COUNCIL_NAME = "Chipata City Council"
CSV_SEP = "|"
EXTRACTED_AT = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

with open(SCHEMAS_PATH, encoding="utf-8") as f:
    SCHEMAS = yaml.safe_load(f)

COLUMNS = [c["name"] for c in SCHEMAS["tables"][TABLE_NAME]["columns"]]
print(f"Documents folder: {DOCS_ROOT}")
print(f"Output: {OUTPUT_DIR / (TABLE_NAME + '.csv')}")

In [ ]:
!pip install pymupdf

## 2. Helper functions

In [ ]:
def parse_amount(value: str | int | float | None) -> float | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, (int, float)):
        return round(float(value), 2)
    cleaned = re.sub(r"[^0-9.]", "", str(value))
    if not cleaned or cleaned == ".":
        return None
    try:
        return round(float(cleaned), 2)
    except ValueError:
        return None


def collection_rate(budgeted: float | None, collected: float | None) -> float | None:
    if budgeted in (None, 0) or collected is None:
        return None
    return round((collected / budgeted) * 100, 2)


def classify_revenue(description: str) -> str:
    desc = description.lower()
    rules = [
        ("market_fees", ["market fee", "market fees", "bus station fee"]),
        ("property_rates", ["residential rate", "commercial rate", "industrial rate", "property rate", "site rent"]),
        ("business_licence", ["licence", "license", "liquor", "occupancy", "retail licence", "wholesale"]),
        ("building_permit", ["permit", "fire certificate", "burial permit", "health permit"]),
        ("levy", ["levy", "personal levy", "business levy", "pole levy", "bicycle levy", "sand levy"]),
        ("fine_penalty", ["penalty", "penalties", "fine"]),
        ("rental_income", ["rental", "lease of council"]),
        ("lgef_transfer", ["local government equalisation", "lgef"]),
        ("cdf_transfer", ["constituency development fund", "cdf funding", "cdf transfer"]),
        ("grant", ["grant in lieu", "other grant", "sector grant", "roads grant", "health grant"]),
    ]
    for source_type, keywords in rules:
        if any(k in desc for k in keywords):
            return source_type
    if "fee" in desc or "charge" in desc:
        return "other"
    return "other"


def make_row(
    record_id: str,
    revenue_year: int,
    description: str,
    source_document: str,
    extraction_method: str,
    collected: float | None = None,
    budgeted: float | None = None,
    reporting_period: str = "annual",
    source_page: int | None = None,
    source_type: str | None = None,
    notes: str = "",
) -> dict:
    return {
        "record_id": record_id,
        "council_name": COUNCIL_NAME,
        "revenue_year": revenue_year,
        "revenue_source_type": source_type or classify_revenue(description),
        "revenue_description": description,
        "budgeted_amount_zmw": budgeted,
        "collected_amount_zmw": collected,
        "collection_rate_percent": collection_rate(budgeted, collected),
        "reporting_period": reporting_period,
        "source_document": source_document,
        "source_page": source_page,
        "extraction_method": extraction_method,
        "extracted_at": EXTRACTED_AT,
        "notes": notes,
    }


def rows_to_dataframe(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    for col in COLUMNS:
        if col not in df.columns:
            df[col] = None
    return df[COLUMNS]


def extract_pdf_text(pdf_path: Path, page_start: int = 1, page_end: int | None = None) -> str:
    """Extract native text from a PDF using PyMuPDF (works when PDF has a text layer)."""
    with pymupdf.open(pdf_path) as doc:
        end = page_end or doc.page_count
        chunks = []
        for i in range(page_start - 1, min(end, doc.page_count)):
            chunks.append(f"=== PAGE {i + 1} ===\n{doc[i].get_text()}")
        return "\n".join(chunks)


def pdf_needs_ocr(pdf_path: Path, min_chars: int = 100) -> bool:
    """Return True if the PDF appears to be a scan (almost no extractable text)."""
    with pymupdf.open(pdf_path) as doc:
        sample = "".join(doc[i].get_text() for i in range(min(3, doc.page_count)))
    return len(sample.strip()) < min_chars


def save_raw_text(pdf_path: Path, text: str) -> Path:
    out = RAW_TEXT_DIR / f"{pdf_path.stem}.txt"
    out.write_text(text, encoding="utf-8")
    return out

## 2b. PDF text extraction and OCR

Two methods are used depending on the PDF:

| Method | When | Library |
|--------|------|---------|
| **Native text** | PDF has a selectable text layer | `pymupdf` |
| **OCR** | PDF is a scanned image | `pdf2image` + `pytesseract` + Tesseract/Poppler |

In [ ]:
# --- OCR configuration (Windows) ---
DEFAULT_TESSERACT = Path(r"C:\Program Files\Tesseract-OCR\tesseract.exe")
DEFAULT_POPPLER_ROOT = Path(
    r"C:\Users\Windows\AppData\Local\Microsoft\WinGet\Packages"
)


def find_tesseract() -> Path | None:
    found = shutil.which("tesseract")
    if found:
        return Path(found)
    return DEFAULT_TESSERACT if DEFAULT_TESSERACT.exists() else None


def find_poppler_bin() -> Path | None:
    found = shutil.which("pdftoppm")
    if found:
        return Path(found).parent
    if DEFAULT_POPPLER_ROOT.exists():
        for pkg in sorted(DEFAULT_POPPLER_ROOT.glob("oschwartz10612.Poppler_*")):
            for sub in pkg.glob("poppler-*/Library/bin"):
                if (sub / "pdftoppm.exe").exists():
                    return sub
    return None


def ocr_pdf(pdf_path: Path, dpi: int = 250, max_pages: int | None = None) -> str:
    """Convert PDF pages to images, then run Tesseract OCR."""
    from pdf2image import convert_from_path
    import pytesseract

    poppler = find_poppler_bin()
    tesseract = find_tesseract()
    if poppler is None:
        raise RuntimeError("Install Poppler: winget install oschwartz10612.Poppler")
    if tesseract is None:
        raise RuntimeError("Install Tesseract: winget install UB-Mannheim.TesseractOCR")

    pytesseract.pytesseract.tesseract_cmd = str(tesseract)
    images = convert_from_path(
        str(pdf_path), dpi=dpi, poppler_path=str(poppler), last_page=max_pages
    )
    return "\n\n".join(
        f"=== PAGE {i + 1} ===\n{pytesseract.image_to_string(img)}"
        for i, img in enumerate(images)
    )


def get_document_text(pdf_path: Path, use_ocr: bool = False) -> tuple[str, str]:
    """Try native text first; fall back to OCR when requested."""
    text = extract_pdf_text(pdf_path)
    if len(text.strip()) > 100:
        return text, "native_text"
    if use_ocr:
        return ocr_pdf(pdf_path), "ocr"
    return text, "ocr_needed"


print(f"Tesseract: {find_tesseract()}")
print(f"Poppler:   {find_poppler_bin()}")

In [ ]:
# --- Parser for OCR budget execution tables (2024 semi-annual report) ---
OCR_AMOUNT_RE = re.compile(r"[\($]?(\d{1,3}(?:,\d{3})+|\d+)\.\d{2}[\)]?")
OCR_CODE_RE = re.compile(r"^[\[\(]?(\d{5,6})")
OCR_CODE_PREFIXES = ("151", "152", "153", "154", "155", "156", "157", "738", "130", "159")


def _ocr_parse_amounts(line: str) -> list[float]:
    out = []
    for m in OCR_AMOUNT_RE.finditer(line):
        try:
            out.append(round(float(m.group(1).replace(",", "")), 2))
        except ValueError:
            pass
    return out


def _ocr_pick_collected(amounts: list[float]) -> float:
    if len(amounts) < 2:
        return amounts[-1]
    budget = amounts[0]
    tol = max(1000.0, budget * 0.02)
    for i in range(1, len(amounts)):
        for j in range(i + 1, len(amounts)):
            if abs(budget - amounts[i] - amounts[j]) <= tol:
                return min(amounts[i], amounts[j])
    below = [a for a in amounts[1:] if a < budget]
    return max(below) if below else amounts[1]


def _ocr_clean_description(line: str) -> str:
    text = re.sub(r"^[\[\(]?\d{5,6}[\]\)\|_\[]?\s*", "", line.strip())
    text = re.sub(OCR_AMOUNT_RE, "", text)
    text = re.sub(r"[\|\[\]_]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" :-")
    return text[:500]


def parse_budget_execution_ocr(ocr_text: str, year: int, source_file: str, start_seq: int = 1) -> list[dict]:
    """Parse revenue rows from OCR text of the semi-annual budget execution report."""
    rows: list[dict] = []
    seq = start_seq

    for chunk in ocr_text.split("=== PAGE "):
        if not chunk.strip():
            continue
        page_part, _, body = chunk.partition("===")
        try:
            page_num = int(page_part.strip())
        except ValueError:
            page_num = None
        if page_num is not None and page_num > 4:
            continue

        for line in body.splitlines():
            line = line.strip()
            if not line or "EXPENSES/EXPENDITURE" in line.upper():
                break
            m = OCR_CODE_RE.match(line)
            if not m or not m.group(1).startswith(OCR_CODE_PREFIXES):
                continue
            amounts = _ocr_parse_amounts(line)
            if len(amounts) < 2:
                continue
            desc = _ocr_clean_description(line)
            if len(desc) < 3:
                continue
            rows.append(
                make_row(
                    record_id=f"REV-{year}-{seq:03d}",
                    revenue_year=year,
                    description=desc,
                    source_document=source_file,
                    extraction_method="ocr",
                    collected=_ocr_pick_collected(amounts),
                    budgeted=amounts[0],
                    reporting_period="semi_annual",
                    source_page=page_num,
                    notes="Auto-parsed from OCR budget execution table",
                )
            )
            seq += 1
    return rows

In [ ]:
# DEMO: preview text extracted from a PDF (run this to verify extraction works)
DEMO_PDF = DOCS_ROOT / "FINANCIAL-STATEMENT.pdf"  # native text
# DEMO_PDF = DOCS_ROOT / "SEMI-ANNUAL-BUDGET-EXECUTION-REPORT-FOR-2024.pdf"  # needs OCR

if DEMO_PDF.exists():
    needs_ocr = pdf_needs_ocr(DEMO_PDF)
    print(f"File: {DEMO_PDF.name} | needs_ocr={needs_ocr}")
    demo_text, demo_method = get_document_text(DEMO_PDF, use_ocr=needs_ocr)
    save_raw_text(DEMO_PDF, demo_text)
    print(f"Method: {demo_method} | chars: {len(demo_text):,}")
    print("--- preview (first 1500 chars) ---")
    print(demo_text[:1500])

In [ ]:
def build_2024_ocr_rows(start_seq: int = 1) -> list[dict]:
    """Extract 2024 revenue rows from scanned semi-annual budget execution report."""
    pdf_path = DOCS_ROOT / "SEMI-ANNUAL-BUDGET-EXECUTION-REPORT-FOR-2024.pdf"
    if not pdf_path.exists():
        print("2024 OCR source not found")
        return []

    cache = RAW_TEXT_DIR / f"{pdf_path.stem}_ocr.txt"
    if cache.exists() and cache.stat().st_size > 500:
        ocr_text = cache.read_text(encoding="utf-8")
        print(f"Using cached OCR: {cache.name}")
    else:
        print(f"Running OCR on {pdf_path.name} (may take ~1 min)...")
        ocr_text = ocr_pdf(pdf_path)
        cache.write_text(ocr_text, encoding="utf-8")

    rows = parse_budget_execution_rows(
        ocr_text, year=2024, source_file=pdf_path.name,
        make_row=make_row, start_seq=start_seq,
    )
    print(f"Parsed {len(rows)} OCR rows for 2024")
    return rows

## 3. Build rows from PDF extraction (2022, 2023, 2025)

These functions call `parse_financial_statement_year()` in `src/revenue_pdf_parser.py`. **No Kwacha amounts are written in the notebook** — the parser opens each PDF, reads the note pages, and extracts figures from the text.

In [ ]:
FS_2023 = DOCS_ROOT / "FINANCIAL-STATEMENT.pdf"
FS_2022 = DOCS_ROOT / "FINANCIAL-STATEMENTS-FOR-THE-YEAR-ENDED-31TH.pdf"
FS_2025 = DOCS_ROOT / "1.-Chipata-City-Council-Bi-Annual-2025-Draft.pdf"

for p in [FS_2023, FS_2022, FS_2025]:
    print(f"{'OK' if p.exists() else 'MISSING'}: {p.name}")

In [ ]:
# Show native PDF text from 2023 financial statement (page 23 = local taxes note)
if FS_2023.exists():
    page23_text = extract_pdf_text(FS_2023, page_start=23, page_end=23)
    save_raw_text(FS_2023, extract_pdf_text(FS_2023, page_start=21, page_end=34))
    print("Page 23 extract (source for 2023 local tax rows):")
    print(page23_text[:1200])

In [ ]:
def build_2023_revenue_rows() -> list[dict]:
    """Extract 2023 revenue from FINANCIAL-STATEMENT.pdf — amounts parsed from PDF text."""
    pdf = DOCS_ROOT / "FINANCIAL-STATEMENT.pdf"
    if not pdf.exists():
        print(f"Missing: {pdf.name}")
        return []
    rows = parse_financial_statement_year(
        pdf, report_year=2023, source_file=pdf.name, make_row=make_row,
    )
    print(f"Parsed {len(rows)} rows for 2023 from {pdf.name}")
    return rows


def build_2022_revenue_rows(start_seq: int = 1) -> list[dict]:
    """Extract 2022 revenue from FINANCIAL-STATEMENTS-FOR-THE-YEAR-ENDED-31TH.pdf."""
    pdf = DOCS_ROOT / "FINANCIAL-STATEMENTS-FOR-THE-YEAR-ENDED-31TH.pdf"
    if not pdf.exists():
        print(f"Missing: {pdf.name}")
        return []
    rows = parse_financial_statement_year(
        pdf, report_year=2022, source_file=pdf.name, make_row=make_row, seq_start=start_seq,
    )
    print(f"Parsed {len(rows)} rows for 2022 from {pdf.name}")
    return rows


def build_2025_semi_annual_rows(start_seq: int = 1) -> list[dict]:
    """Extract Jan–Jun 2025 cash receipts from bi-annual draft (page 8)."""
    pdf = DOCS_ROOT / "1.-Chipata-City-Council-Bi-Annual-2025-Draft.pdf"
    if not pdf.exists():
        print(f"Missing: {pdf.name}")
        return []
    rows = parse_financial_statement_year(
        pdf, report_year=2025, source_file=pdf.name, make_row=make_row, seq_start=start_seq,
    )
    print(f"Parsed {len(rows)} rows for 2025 from {pdf.name}")
    return rows

## 4. Build and inspect dataset

In [ ]:
all_rows = []
all_rows.extend(build_2023_revenue_rows())
all_rows.extend(build_2022_revenue_rows())
all_rows.extend(build_2024_ocr_rows())      # automated OCR extraction
all_rows.extend(build_2025_semi_annual_rows())

revenue_df = rows_to_dataframe(all_rows)
print(f"Total rows: {len(revenue_df)}")
print(f"Years covered: {sorted(revenue_df['revenue_year'].dropna().unique())}")
revenue_df.groupby(["revenue_year", "revenue_source_type"]).size().unstack(fill_value=0)

In [ ]:
revenue_df.head(15)

In [ ]:
# Rows with both budget and collected amounts (2023 variance note)
revenue_df[revenue_df["budgeted_amount_zmw"].notna()][
    ["record_id", "revenue_year", "revenue_description", "budgeted_amount_zmw", "collected_amount_zmw", "collection_rate_percent"]
]

## 5. Validation

In [ ]:
allowed_types = {
    c["allowed_values"]
    for c in SCHEMAS["tables"][TABLE_NAME]["columns"]
    if c["name"] == "revenue_source_type"
}.pop()

assert list(revenue_df.columns) == COLUMNS, "Column order mismatch"
assert revenue_df["record_id"].is_unique, "Duplicate record_id values"
assert revenue_df["council_name"].eq(COUNCIL_NAME).all()
assert revenue_df["revenue_source_type"].isin(allowed_types).all(), "Invalid revenue_source_type"
assert revenue_df["revenue_description"].str.len().le(500).all()

# Round-trip pipe separator test
test_path = OUTPUT_DIR / "_validate_revenue.csv"
revenue_df.to_csv(test_path, sep=CSV_SEP, index=False, encoding="utf-8")
roundtrip = pd.read_csv(test_path, sep=CSV_SEP)
assert list(roundtrip.columns) == COLUMNS
test_path.unlink(missing_ok=True)

print("All validation checks passed.")

## 6. Export CSV


In [ ]:
output_path = OUTPUT_DIR / f"{TABLE_NAME}.csv"
revenue_df.to_csv(output_path, sep=CSV_SEP, index=False, encoding="utf-8")

print(f"Exported {len(revenue_df)} rows")
print(f"File: {output_path}")
print(f"Separator: '{CSV_SEP}'")
print()
print("Preview:")
print(revenue_df[["record_id", "revenue_year", "revenue_source_type", "revenue_description", "collected_amount_zmw"]].head(8).to_string(index=False))

## 7. Summary statistics 

In [ ]:
summary = (
    revenue_df.groupby("revenue_year")
    .agg(
        rows=("record_id", "count"),
        total_collected_zmw=("collected_amount_zmw", "sum"),
        sources=("revenue_source_type", "nunique"),
    )
    .reset_index()
)
summary

## 8. Limitations

Chipata's financial notes use multi-column tables. PyMuPDF often returns labels and amounts in the wrong order. The parser uses heuristics (regex, page splits, OCR cleanup like r,594,400 -> 1,594,400), so some rows can still be mis-paired even though amounts are extracted from the document, not typed in.

## C. CDF PROJECTS DATA EXTRACTION SCRIPTS

## CSC 4792 — Chipata City Council CDF Projects Dataset

**Course:** CSC 4792 Data Mining and Warehousing (2025/26)  
**Institution:** University of Zambia  
**Council:** Chipata City Council   
**Constituencies:** Chipata Central, Luangeni  
**Dataset file:** `outputs/db-unza26-csc4792-chipata_city_council_cdf_projects.csv`  
**Format:** CSV with pipe (`|`) separator  

---

## 1. Business Understanding (CRISP-DM)

**Problem:** Chipata City Council publishes CDF (Constituency Development Fund) project information across multiple official PDF documents. These documents are unstructured and difficult to analyse directly.

**Objective:** Build a structured, evidence-based dataset where each row is one CDF project and every value can be traced to an official source document.

**Scope:** CDF community projects only (approved, proposed, and not prioritized), 2022–2025.

**Data ethics rule:** No values are invented. If a field is missing in the source PDF, it remains blank in the CSV.


## 2. Methodology Overview (CRISP-DM) 

| Phase | Activity in this project | Notebook section |
|-------|-------------------------|------------------|
| Business Understanding | Define CDF tracking problem and dataset scope | Section 1 |
| Data Understanding | Identify sources, assess PDF quality, inspect outputs | Sections 6–7 |
| Data Preparation | Transcribe, structure, integrate, export CSV | Sections 8–9 |
| Exploratory analysis | Summary statistics and sample filters | Sections 10–12 |

**Data collection approach:** All source PDFs were **downloaded from the official council document portal** and saved locally in `csc_4792_dataset_docs/` before any extraction. This notebook documents how those local files were processed into a structured CSV.


## 3. Data Collection — Official Portal Downloads

### Step 1 — Download from official portal
Official Chipata City Council CDF documents were downloaded from the council document portal and placed in:

`csc_4792_dataset_docs/CDF Approved Project/`

### Step 2 — Keep raw files unchanged
Downloaded PDFs were stored without modification to preserve evidence and reproducibility.

### Step 3 — Extract locally
Because many PDFs are scanned images, data was extracted locally using Python (PyMuPDF) and manual transcription from rendered page images. No live website scraping was performed at extraction time.

## 4. Dataset Schema (24 Columns)

| Column | Description |
|--------|-------------|
| `record_id` | Unique identifier |
| `council_name` | Chipata City Council |
| `constituency` | Chipata Central or Luangeni |
| `cdf_year` | 2022–2025 |
| `project_name` | Short title from source |
| `project_description` | Full description from source |
| `project_sector` | Health, Education, Roads, etc. |
| `ward` | Ward name |
| `engineer_estimate_zmw` | Amount in ZMW (blank if not in source) |
| `approval_status` | Approved / Not Approved / Not Prioritized / Unknown |
| `status_justification` | Reason when not approved |
| `source_document` | Original PDF filename |
| `source_page` | Page number in PDF |
| `source_document_category` | Document type |
| `notes` | Transcription and quality notes |

**Categories:** `approved_projects_table`, `approval_notification`, `wdc_proposed`, `disapproved_list`, `cdf_progress_report`, `approved_community_projects`


## 5. Source Documents Used (527 Records)

| Source PDF | Records | Constituency | Year(s) |
|------------|---------|--------------|---------|
| CHIPATA-CENTRAL-2022.pdf | 29 | Chipata Central | 2022 |
| 2022-LAUNGENI.pdf (pages 4–5) | 20 | Luangeni | 2022 |
| CHIPATA-CENTRAL-2023.pdf | 23 | Chipata Central | 2023 |
| luangeni-2023.pdf (2023 pages) | 27 | Luangeni | 2023 |
| CHIPATA-CENTRAL-2024.pdf | 12 | Chipata Central | 2024 |
| luangeni-2023.pdf (2024 pages) | 23 | Luangeni | 2024 |
| 2024_CDF1.pdf | 35 | Both | 2024 |
| 2024-WDC-PROPOSED-COMMUNITY-PROJECTS-CHIPATA.pdf | 87 | Chipata Central | 2024 |
| 2024-WDC-PROPOSED-COMMUNITY-PROJECTS-LUANGENI.pdf | 74 | Luangeni | 2024 |
| APPROVED-COMMUNITY-PROJECTS-CHIPATA-CENTRAL.pdf | 15 | Chipata Central | 2025 |
| APPROVED-COMMUNITY-PROJECTS-2025-LUANGENI.pdf | 12 | Luangeni | 2025 |
| 2025-WARD-DEVELOPMENT-COMMITTEES-PROPOSED.pdf | 73 | Chipata Central | 2025 |
| 2025-DISAPPROVED-PROJECTS-WITH-JUSTIFICATIONS.pdf | 97 | Both | 2025 |

In [ ]:
from pathlib import Path

# Resolve project root whether kernel cwd is project root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent]:
    if (candidate / "csc_4792_dataset_docs").exists():
        PROJECT_ROOT = candidate
        break

CDF_SOURCE_DIR = PROJECT_ROOT / "csc_4792_dataset_docs" / "CDF Approved Project"

pdf_files = sorted(CDF_SOURCE_DIR.glob("*.pdf"))
print("Official portal PDFs in local folder:", len(pdf_files))
for pdf in pdf_files:
    print(" -", pdf.name)

## 6. Data Understanding — PDF Quality Assessment

**Purpose (CRISP-DM):** Determine whether PDF pages contain selectable text or are scanned images.

**Method:** PyMuPDF `page.get_text()` character count per page.

**Finding:** Most CDF PDFs are scanned; manual transcription from rendered PNGs is required.

In [ ]:
import pymupdf

sample_pdf = CDF_SOURCE_DIR / "APPROVED-COMMUNITY-PROJECTS-CHIPATA-CENTRAL.pdf"
doc = pymupdf.open(sample_pdf)

print("Sample file:", sample_pdf.name)
print("Pages:", doc.page_count)
print("-" * 50)
for i in range(doc.page_count):
    n = len(doc[i].get_text("text").strip())
    kind = "scanned" if n == 0 else "has text"
    print(f"Page {i+1}: {n} characters -> {kind}")
doc.close()


## 7. Data Extraction Pipeline

| Step | Action | Method |
|------|--------|--------|
| 1 | Download PDFs from official portal | Manual download to local folder |
| 2 | Inventory files | `Path.glob("*.pdf")` |
| 3 | Assess page quality | PyMuPDF text check |
| 4 | Render pages to PNG | `get_pixmap(dpi=300)` → `data/discovery/` |
| 5 | Transcribe rows | Manual reading of PNG tables |
| 6 | Structure records | `make_record()` / `wdc_rows()` in Python modules |
| 7 | Integrate and export | `build_dataset()` → pipe-separated CSV |

**Design decisions:** no deduplication across documents; preserve source spelling; blank missing values.

In [ ]:
import pymupdf

DISCOVERY_DIR = PROJECT_ROOT / "data" / "discovery"
DISCOVERY_DIR.mkdir(parents=True, exist_ok=True)

render_pdf = CDF_SOURCE_DIR / "CHIPATA-CENTRAL-2022.pdf"
doc = pymupdf.open(render_pdf)
for i in range(doc.page_count):
    out = DISCOVERY_DIR / f"{render_pdf.stem}__p{i+1:02d}.png"
    if not out.exists():
        doc[i].get_pixmap(dpi=300).save(str(out))
    print("Saved:", out.name)
doc.close()

### 7.1 Example structured record from one transcribed PDF row

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src" / "extraction"))
from build_cdf_dataset import make_record

example = make_record(
    record_id="CCC-2025-001",
    constituency="Chipata Central",
    cdf_year=2025,
    source_row_number=1,
    project_name="Industrial Boreholes",
    project_description="Construction of Industrial Boreholes at Kabulonga in Kanjala, Mchini in Dilika and Chimwemwe in Kapata",
    project_sector="Health",
    ward="All",
    project_site_location="Kabulonga (Magazine), Mchini and Chimwemwe",
    engineer_estimate_zmw=8500000.00,
    approval_status="Approved",
    source_document_category="approved_projects_table",
    source_document="APPROVED-COMMUNITY-PROJECTS-CHIPATA-CENTRAL.pdf",
    source_page=2,
)
print("record_id:", example["record_id"])
print("source_document:", example["source_document"])
print("engineer_estimate_zmw:", example["engineer_estimate_zmw"])

## 8. Data Preparation — Build and Export Final CSV

Integrates all transcribed sources into one 527-row dataset and exports the assignment CSV.

In [ ]:
import sys
import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT / "src" / "extraction"))
from build_cdf_dataset import build_dataset

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "combined_raw_cdf_projects.csv"
RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
FINAL_CSV = OUTPUT_DIR / "db-unza26-csc4792-chipata_city_council_cdf_projects.csv"

df = build_dataset()
df.to_csv(RAW_CSV, sep="|", index=False)

try:
    df.to_csv(FINAL_CSV, sep="|", index=False)
    print("Saved:", FINAL_CSV)
except PermissionError:
    print("Note: close the CSV in Excel, then re-run this cell.")
    print("Dataset saved to:", RAW_CSV)

print("Shape:", df.shape)
print("Years:", sorted(df["cdf_year"].unique()))
print("Constituencies:\n", df["constituency"].value_counts())


## 9. Load and Inspect Final Dataset


In [ ]:
# df is built in the previous cell; reload from raw CSV if running this cell alone
try:
    df
except NameError:
    import pandas as pd
    df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "combined_raw_cdf_projects.csv", sep="|")

print("Shape:", df.shape)
display(df.head(10))
print("By year and constituency:")
print(df.groupby(["cdf_year", "constituency"]).size())


## 10. Data Quality Validation

In [ ]:
cc25 = df[
    (df["constituency"] == "Chipata Central")
    & (df["cdf_year"] == 2025)
    & (df["source_document_category"] == "approved_projects_table")
]
total = round(cc25["engineer_estimate_zmw"].sum(), 2)
print("CC 2025 approved records:", len(cc25))
print("Total ZMW:", total)
print("Expected: 20916106.93")
print("Match:", total == 20916106.93)
print("\nBy source category:")
print(df["source_document_category"].value_counts())
print("\nMissing amounts:", df["engineer_estimate_zmw"].isna().sum(), "of", len(df))


## 11. Sample Exploratory Filters

In [ ]:
health = df[(df["approval_status"] == "Approved") & (df["project_sector"] == "Health")]
print("Approved Health projects:", len(health))
display(health[["record_id", "constituency", "cdf_year", "project_name", "ward"]].head(10))


## 12. Limitations

1. **Scanned PDFs:** Portal downloads are mostly image scans; data was manually transcribed from rendered PNGs.
2. **Missing amounts:** WDC and disapproved lists often omit costs — field left blank.
3. **Duplicate appearances:** Same project may appear in multiple documents; separate rows kept for provenance.
4. **Portal snapshot:** Data reflects documents available at download time.
5. **Scope:** CDF projects only — not budgets, LGEF, or IDPs.
6. **Excluded files:** Blank scans and duplicate documents omitted.
7. **Transcription risk:** Minor human error possible; mitigated by source page references.

## 13. Reproducibility

From the project root folder:

```powershell
cd src/extraction
python build_cdf_dataset.py
```

Or run all code cells in this notebook in order. Close the CSV in Excel before regenerating.